# Test model

In [ ]:
from transformers import AutoTokenizer
import requests
import json
import re
import time

url = "http://xxxx:8000/v1/chat/completions"
headers = {"Content-Type": "application/json"}
model_addr = "/path/to/model"
tokenizer = AutoTokenizer.from_pretrained(model_addr)

user_input = f"hello"
messages = [
    {
        "role": "system",
        "content": "You are a human.",
    },
    {"role": "user", "content": user_input},
]

# compute tokenizers
prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
encoding = tokenizer(prompt_text, return_tensors='pt')
tot_tokens = 0
for iid, input_ids in enumerate(encoding.input_ids):
    num_tokens = len(input_ids)
    tot_tokens += num_tokens
    print(f"Number of tokens in the prompt of input id {iid}: {num_tokens}")

# start send request
data = {
    "model": "deepseek70b",
    "messages": messages,
    "temperature": 0.2,
    "top_p": 0.9,
    "repetition_penalty": 1.2,
}
start_time = time.perf_counter()
response = requests.post(url, headers=headers, data=json.dumps(data))
end_time = time.perf_counter()
elapsed_time = end_time - start_time
print(f"Elapsed {tot_tokens} in {elapsed_time:.6f} seconds")
print(f"response json: {response.json()}")
output_text = response.json()['choices'][0]['message']['content']
print(f"output: {output_text}")

# output tokens
output_tokens = len(tokenizer.encode(output_text))
print(f"Number of tokens in the output: {output_tokens}")
tot_tokens += output_tokens

print(f"avg tokens/s: {tot_tokens / elapsed_time}")